# Geolocation Processing

This notebook processes a dataset to extract location information and convert it to geographic coordinates (latitude/longitude) using Ollama for location extraction and OpenStreetMap (Nominatim) for geocoding.

## 1. Import Required Libraries

In [23]:
import pandas as pd
import numpy as np
import requests
import os
import glob
import json
import time
from datetime import datetime
from pathlib import Path
import re
from typing import Optional, Dict, Tuple

from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [24]:
# pip install geopy folium requests

## 2. Load Latest Dataset from Static Folder

In [25]:
static_folder = Path("../../static")

PREFERRED_INPUT = "all_merged.csv"
EXCLUDE_FILES = {"geocoded_data.csv"}

preferred_path = static_folder / PREFERRED_INPUT
if preferred_path.exists():
    latest_file = preferred_path
else:
    csv_files = [p for p in static_folder.glob("*.csv") if p.name not in EXCLUDE_FILES]
    if not csv_files:
        raise FileNotFoundError(f"No usable CSV files found in {static_folder.absolute()}")
    latest_file = max(csv_files, key=lambda p: p.stat().st_mtime)

print(f"Latest file found: {latest_file.name}")
print(f"Modified: {datetime.fromtimestamp(latest_file.stat().st_mtime)}")

df = pd.read_csv(latest_file)

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")
print("First few rows:")
print(df.head())


Latest file found: all_merged.csv
Modified: 2026-02-11 10:43:31.373075

Dataset shape: (2184, 15)
Columns: ['Date', 'ExtractedAction', 'ExtractedAge', 'ExtractedDate', 'ExtractedGender', 'ExtractedTime', 'KeywordExtracted', 'KeywordMatch', 'Location', 'RightWingRelated', 'SourceFile', 'Text', 'Title', 'Topic', 'URL']

First few rows:
         Date                   ExtractedAction ExtractedAge ExtractedDate  \
0  18.12.2025  ['schlagen', 'körperverletzung']           []    18.12.2025   
1  18.07.2019                                []       ['34']    18.07.2019   
2  26.07.2019                                []           []    26.07.2019   
3  19.07.2021                   ['beleidigung']       ['28']    19.07.2021   
4  11.10.2019                                []           []    11.10.2019   

  ExtractedGender ExtractedTime      KeywordExtracted          KeywordMatch  \
0        ['mann']            []  ['fremdenfeindlich']  ['fremdenfeindlich']   
1              []     ['22.00']   ['v

In [26]:
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)
geocoded_file = output_dir / "geocoded_data.csv"

df_previous = None
if geocoded_file.exists():
    print(f"\n Loading previously geocoded data from: {geocoded_file}")
    df_previous = pd.read_csv(geocoded_file)
    print(f"  Found {len(df_previous)} previously geocoded rows")
    
    if 'URL' in df.columns and 'URL' in df_previous.columns:
        df = df.merge(
            df_previous[['URL', 'standardized_location', 'ollama_extraction', 'latitude', 'longitude']],
            on='URL', how='left', suffixes=('_new', '')
        )
        for col in ['standardized_location', 'ollama_extraction', 'latitude', 'longitude']:
            if col + '_new' in df.columns:
                df[col] = df[col].fillna(df[col + '_new'])
                df = df.drop(columns=[col + '_new'], errors='ignore')
    
    print(f"  After merging: {df['latitude'].notna().sum()} rows already have coordinates\n")
else:
    print(f"\n  No previous geocoding found at {geocoded_file}")
    print("  Starting fresh...\n")



 Loading previously geocoded data from: output/geocoded_data.csv
  Found 2082 previously geocoded rows
  After merging: 2082 rows already have coordinates



## 3. Ollama Configuration

In [27]:
def save_checkpoint(dataframe, message=""):
    output_dir = Path("./output")
    output_dir.mkdir(exist_ok=True)
    output_file = output_dir / "geocoded_data.csv"
    
    df_geocoded = dataframe[dataframe['latitude'].notna() & dataframe['longitude'].notna()].copy()
    
    if len(df_geocoded) > 0:
        output_columns = df_geocoded.columns.tolist()
        geocoding_cols = ['standardized_location', 'latitude', 'longitude', 'ollama_extraction']
        for col in geocoding_cols:
            if col in output_columns:
                output_columns.remove(col)
                output_columns.append(col)
        
        df_output = df_geocoded[output_columns]
        df_output.to_csv(output_file, index=False)
        print(f"  ✓ Checkpoint saved: {len(df_output)} rows with coordinates {message}")
    else:
        print(f"  (No coordinates yet to save) {message}")


In [28]:
OLLAMA_API_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "gemma3:latest" 
REQUEST_TIMEOUT = 30

def test_ollama_connection():
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json().get('models', [])
            print(f"✓ Ollama is running with {len(models)} model(s):")
            for model in models:
                print(f"  - {model.get('name', 'Unknown')}")
            return True
        else:
            print("✗ Ollama returned an error status")
            return False
    except requests.exceptions.ConnectionError:
        print("✗ Cannot connect to Ollama. Make sure it's running on http://localhost:11434")
        return False
    except Exception as e:
        print(f"✗ Error testing Ollama: {e}")
        return False

print("Testing Ollama connection...")
ollama_available = test_ollama_connection()

Testing Ollama connection...
✓ Ollama is running with 6 model(s):
  - qwen3:1.7b
  - llava:latest
  - smollm2:latest
  - gemma3:latest
  - gemma2:2b
  - codellama:latest


## 4. Extract and Standardize Location Data with Ollama

In [29]:
def standardize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text) 
    return text

def extract_location_with_ollama(text: str, location_context: str = None, row_context: Dict = None) -> Optional[str]:
    if not ollama_available or not text or not isinstance(text, str):
        return None
    
    context_info = ""
    if location_context:
        context_info = f"Geographic context (city, district): {location_context}\n"
    
    prompt = f"""{context_info}Extract the MOST SPECIFIC location mentioned in this text.

PRIORITY - Look for:
1. Street names (e.g., "Ratsgasse", "Friedrich-Wolf-Ring", "Berliner Straße")
2. Landmarks, buildings, or specific places (e.g., "Bahnhof", "Bushaltestelle", "Schule")
3. Areas or neighborhoods
4. Addresses with numbers

Format: [Street/Place], [City], [District] (if available)

Examples of correct responses:
- "Ratsgasse, Velten, Oberhavel"
- "Mühlenweg 12, Strausberg, Märkisch-Oderland"
- "Bernauer Straße, Oranienburg, Oberhavel"
- "corner near Bahnhof, Neuruppin, Ostprignitz-Ruppin"

Return ONLY the location, nothing else. If no specific location found in text, use context: "{location_context}". If completely unknown, respond 'UNKNOWN'.

Text: "{text}"

Most specific location:"""
    
    try:
        response = requests.post(
            OLLAMA_API_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=REQUEST_TIMEOUT
        )
        
        if response.status_code == 200:
            result = response.json()
            location = result.get("response", "").strip()
            location = standardize_text(location)
            
            if location and location.lower() != "unknown":
                if len(location) > 120:
                    location = location[:120].rsplit(',', 1)[0].strip()
                
                return location
    except Exception as e:
        print(f"Error querying Ollama: {e}")
    
    return None


if 'standardized_location' not in df.columns:
    df['standardized_location'] = None
if 'ollama_extraction' not in df.columns:
    df['ollama_extraction'] = None

already_extracted = df['standardized_location'].notna().sum()
print(f"\nExtracting precise, concise locations from {len(df)} rows...")
print(f"Already extracted: {already_extracted} locations")
print("Note: This may take a while depending on dataset size and Ollama response time.\n")


Extracting precise, concise locations from 2184 rows...
Already extracted: 2080 locations
Note: This may take a while depending on dataset size and Ollama response time.



In [31]:
sample_size = 2100
shuffle_data = True

def needs_extraction(row) -> bool:
    val = row.get('standardized_location') or row.get('ollama_extraction')
    return pd.isna(val) or str(val).strip().lower() in {'', 'nan', 'none', 'unknown'}

if shuffle_data:
    df_process = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Processing {sample_size} rows out of {len(df)} total rows (shuffled)...\n")
else:
    df_process = df.copy()
    print(f"Processing {sample_size} rows out of {len(df)} total rows (sequential)...\n")

for idx, row in df_process.head(sample_size).iterrows():
    if not needs_extraction(row):
        continue

    location_context = str(row.get('Location') or '')
    location_text = ' '.join([
        str(row.get('Title') or ''),
        str(row.get('Text') or ''),
    ]).strip()

    if location_text:
        extracted_location = extract_location_with_ollama(location_text, location_context, dict(row))
        original_idx = df[df['URL'] == row.get('URL')].index[0] if 'URL' in df.columns and pd.notna(row.get('URL')) else None
        if extracted_location and original_idx is not None:
            df.at[original_idx, 'ollama_extraction'] = extracted_location
            df.at[original_idx, 'standardized_location'] = extracted_location

        if extracted_location:
            print(f"  Row {idx + 1}: {extracted_location} ({row.get('Location', 'Unknown')})")

    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx + 1}/{sample_size} rows...")

    time.sleep(0.5)

print(f"\nLocation extraction complete!")
print(f"Successfully extracted: {df['standardized_location'].notna().sum()} locations")
print("Saving checkpoint...")
save_checkpoint(df, "(after location extraction)")
print("\nSample results:")
print(df[['standardized_location', 'ollama_extraction']].head(10))


Processing 2100 rows out of 2184 total rows (shuffled)...

  Row 549: BAB 13 bei Duben (DS), Bronkow (OSL), Überregional (BAB 13 bei Duben (DS), Bronkow (OSL), Überregional)
  Row 582: Block T, Berlin, Berlinweit (berlinweit)
  Row 633: Corinthstraße, Friedrichshain, Friedrichshain-Kreuzberg/Lichtenberg U-Bahnhof Frankfurter Allee, Lichtenberg (bezirksübergreifend)
  Row 709: Reichstagswiese, Berlin, Berlinweit (berlinweit)
  Row 713: Gedenkstätten und Mahnmalen, Berlinweit (berlinweit)
  Row 744: Treskowallee, Berlin, Berlin (bezirksübergreifend)
  Row 766: Alexanderplatz, Mitte, Friedrichshain-Kreuzberg (bezirksübergreifend)
  Row 773: Freilichtbühne, Altstadt, Marienberg (Altstadt, Marienberg, Brandenburg an der Havel)
  Row 777: Erich-Kästner-Straße, Treptow-Köpenick, Berlin (nan)
  Row 780: Glogauer Straße, Berlin, Berlin (bezirksübergreifend)
  Processed 780/2100 rows...
  Row 786: rund 90 angezeigten Versammlungen, berlinweit (berlinweit)
  Row 800: BRANDENBURG, POTSDAM, ÜBERREG

## 5. Geocode Locations with OpenStreetMap

In [35]:
REGEOCODE_IMPRECISE = True
REGEOCODE_ALL = True
IMPRECISE_LEVELS = {"district", "city", "region", "unknown"}

In [36]:
geocoder = Nominatim(user_agent="geolocation_processor")

def build_query(row):
    base = row['standardized_location'] if pd.notna(row['standardized_location']) else row.get('ollama_extraction')
    if not pd.notna(base):
        return None
    base = standardize_text(str(base))
    context = standardize_text(str(row.get('Location') or ''))
    q = base
    if context and context.lower() not in q.lower():
        q = f"{q}, {context}"
    if 'germany' not in q.lower():
        q = f"{q}, Germany"
    return q

def geocode_location(query: str) -> Optional[Tuple[float, float]]:
    if not query or not isinstance(query, str):
        return None
    try:
        location = geocoder.geocode(query, timeout=10, country_codes='de')
        if location:
            return (location.latitude, location.longitude)
    except GeocoderTimedOut:
        print(f"  Timeout geocoding: {query}")
    except GeocoderServiceError as e:
        print(f"  Service error geocoding {query}: {e}")
    except Exception as e:
        print(f"  Error geocoding {query}: {e}")
    return None

# Initialize coordinates if they don't exist
if 'latitude' not in df.columns:
    df['latitude'] = None
if 'longitude' not in df.columns:
    df['longitude'] = None

already_geocoded = df[df['latitude'].notna() & df['longitude'].notna()].shape[0]
if REGEOCODE_ALL:
    to_geocode = df['standardized_location'].notna().shape[0]
else:
    to_geocode = df[(df['standardized_location'].notna()) & (df['latitude'].isna())].shape[0]

print(f"Geocoding locations...")
print(f"  Already geocoded: {already_geocoded}")
print(f"  To geocode: {to_geocode}\n")

geocoded_count = 0
for idx, row in df.iterrows():
    has_coords = pd.notna(row['latitude']) and pd.notna(row['longitude'])
    if has_coords and not REGEOCODE_ALL:
        continue

    query = build_query(row)
    if not query:
        continue

    coords = geocode_location(query)
    if coords:
        df.at[idx, 'latitude'] = coords[0]
        df.at[idx, 'longitude'] = coords[1]
        geocoded_count += 1

    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx + 1} rows ({geocoded_count} newly geocoded)...")

    if geocoded_count > 0 and geocoded_count % 50 == 0:
        save_checkpoint(df, f"(after {geocoded_count} geocoded)")

    time.sleep(1)

print(f"\nFirst pass complete: {geocoded_count} locations geocoded")
print("Saving checkpoint...")
save_checkpoint(df, "(after first geocoding pass)")

print(f"\nGeocoding complete!")
print(f"Total successfully geocoded overall: {df[df['latitude'].notna()].shape[0]}")
print("\nSample results with coordinates:")
print(df[df['latitude'].notna()][['standardized_location', 'Location', 'latitude', 'longitude']].head(10))

Geocoding locations...
  Already geocoded: 2082
  To geocode: 2184

  Processed 10 rows (6 newly geocoded)...
  Processed 20 rows (12 newly geocoded)...
  Processed 30 rows (20 newly geocoded)...
  Processed 40 rows (28 newly geocoded)...
  Processed 50 rows (32 newly geocoded)...
  Processed 60 rows (39 newly geocoded)...
  Processed 70 rows (43 newly geocoded)...
  ✓ Checkpoint saved: 2082 rows with coordinates (after 50 geocoded)
  Processed 80 rows (50 newly geocoded)...
  ✓ Checkpoint saved: 2082 rows with coordinates (after 50 geocoded)
  Processed 90 rows (54 newly geocoded)...
  Processed 100 rows (59 newly geocoded)...
  Processed 110 rows (62 newly geocoded)...
  Processed 120 rows (68 newly geocoded)...
  Processed 130 rows (76 newly geocoded)...
  Processed 140 rows (83 newly geocoded)...
  Processed 150 rows (87 newly geocoded)...
  Processed 160 rows (93 newly geocoded)...
  ✓ Checkpoint saved: 2082 rows with coordinates (after 100 geocoded)
  Processed 170 rows (101 newl

## 6. Store Results and Geocoordinates

In [37]:
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "geocoded_data.csv"

df_geocoded = df[df['latitude'].notna() & df['longitude'].notna()].copy()

output_columns = df_geocoded.columns.tolist()
geocoding_cols = ['standardized_location', 'latitude', 'longitude', 'ollama_extraction']
for col in geocoding_cols:
    if col in output_columns:
        output_columns.remove(col)
        output_columns.append(col)

df_output = df_geocoded[output_columns]
df_output.to_csv(output_file, index=False)
print(f"✓ Results saved to: {output_file}")
print(f"  Total rows with coordinates: {len(df_output)}")
print(f"  Rows skipped (no geocoding): {len(df) - len(df_output)}")

print("\n" + "="*60)
print("GEOCODING SUMMARY")
print("="*60)
print(f"Total locations extracted: {df['standardized_location'].notna().sum()}")
print(f"Successfully geocoded: {df_geocoded.shape[0]}")
print(f"Success rate: {(len(df_output) / df['standardized_location'].notna().sum() * 100):.1f}%" if df['standardized_location'].notna().sum() > 0 else "No locations found")

if len(df_output) > 0:
    print("\nCoordinate ranges:")
    print(f"  Latitude: {df_output['latitude'].min():.4f} to {df_output['latitude'].max():.4f}")
    print(f"  Longitude: {df_output['longitude'].min():.4f} to {df_output['longitude'].max():.4f}")

print("\nSample geocoded results:")
print(df_output[['standardized_location', 'latitude', 'longitude']].head(10))

✓ Results saved to: output/geocoded_data.csv
  Total rows with coordinates: 2082
  Rows skipped (no geocoding): 102

GEOCODING SUMMARY
Total locations extracted: 2176
Successfully geocoded: 2082
Success rate: 95.7%

Coordinate ranges:
  Latitude: 48.7721 to 53.4104
  Longitude: 6.9537 to 14.7154

Sample geocoded results:
                               standardized_location   latitude  longitude
0                         Ströbitz, Cottbus, Cottbus  51.758399  14.299960
1            Bollwerk, Neuruppin, Ostprignitz-Ruppin  52.913626  12.809695
2                       Elbanleger, Lenzen, Prignitz  53.090801  11.474862
3  Waldfriedhofes, Großräschen, Oberspreewald-Lau...  51.588327  14.012314
4        Mühlenweg 12, Strausberg, Märkisch-Oderland  52.578431  13.890798
5        Erich-Mühsam-Straße, Oranienburg, Oberhavel  52.749293  13.238641
6                      Lübbensee, Templin, Uckermark  53.119350  13.500556
7            Bernauer Straße, Oranienburg, Oberhavel  52.753886  13.237345
8 